# H2O Implementation -- official H2O engine (submodule) + KVQuant-family harness -- 60% budget

This notebook evaluates H2O using the **official FMInference/H2O code from the
`H2O` submodule of the KVCacheCompression repo** -- specifically
`h2o_hf/utils_real_drop/modify_llama.py`'s `H2OKVCache_LayerWise`, the
authors' real-KV-dropping eviction engine (per-layer AND per-head heavy-hitter
scores, physical pruning of the cache tensors) -- inside the exact same
experimental harness as the KVQuant-family notebooks
(`KVQuant_2bit/3bit/4bit_Implementation.ipynb`,
`KVQuant_Baseline_Implementation.ipynb`): same model, same pinned package
versions, same seed, same dataset loading/tokenization/chunking, same GSM8K
prompt + question set + answer grading, same metric definitions, same
result-CSV schema.

This is one of three H2O eviction-budget variants (20% / 40% / 60% total
cache budget) -- identical in every other respect to
`H2O_official_submodule_Implementation.ipynb` (the 40% budget notebook) and
`H2O_official_submodule_20pct_Implementation.ipynb`. Only `HEAVY_RATIO` /
`RECENT_RATIO` and `METHOD_NAME` differ between the three.

**What comes from the submodule vs. what is harness code:**

- FROM THE SUBMODULE (unmodified): `H2OKVCache_LayerWise` -- heavy-hitter
  score accumulation (`_update_hh_score`: attention mass summed over batch
  and query rows, accumulated per head per cached position), the keep-set
  selection (per-head top-`hh_size` heavy hitters + last `recent_size`
  recent tokens), and the physical KV dropping. One engine instance per
  layer, exactly as `H2OLlamaAttention` instantiates it in the official
  code.
- HARNESS CODE (this notebook): the loop that runs the stock TinyLlama with
  `output_attentions=True` and feeds each layer's attention weights + legacy
  KV tuple through that layer's engine after every forward. The submodule's
  full `H2OLlamaAttention` module replacement could not be used directly
  because it targets an older transformers API (no `cache_position`, legacy
  tuple plumbing) and **assumes MHA** -- its cache-vs-score shapes break on
  GQA models like TinyLlama (4 KV heads vs. 32 query heads). Under the
  pinned `transformers==4.43.4` with eager attention, the stock model
  computes the identical attention math that class reimplements, so driving
  the official engine from the stock model's attention outputs preserves the
  method exactly.
- ONE GQA ADAPTATION (in harness code, submodule untouched): the engine
  expects one score row per cached KV head. TinyLlama's 32 query heads share
  4 KV heads (groups of 8), so each layer's attention weights are reduced to
  KV heads by summing the 8 query heads in each group -- the total attention
  mass received by that KV head's cache entries, the natural GQA
  generalization of the paper's per-head scores.

**Method requirements that legitimately differ from the KVQuant notebooks:**

1. `attn_implementation="eager"` -- H2O needs real attention weights;
   FlashAttention/SDPA cannot return them. Latency gaps vs. the sdpa
   notebooks therefore mix "eager tax" with "H2O tax"; run the baseline
   notebook once with eager to decompose them.
2. Memory here is **measured** (the engine physically drops KV positions);
   the quantized KVQuant notebooks report **calculated** bytes (simulated
   compression). Keep that distinction in mind when comparing memory columns.

**Budgets:** the official engine takes absolute `hh_size`/`recent_size`
counts (as in the repo's `run_summarization.py`). They are derived per sample
from ratios: `hh_size = int(0.30 * total_len)`, `recent_size = int(0.30 *
total_len)` -- i.e. a **60% total budget** (1.5x the other two H2O
notebooks' 40%, and 3x the 20% variant), evenly split between the
heavy-hitter and recency components, matching that same even-split
convention. Fresh engines (fresh scores) are created per sample, mirroring
the official per-request `_clean_scores` usage.

**Harness parity (identical to the KVQuant family):** PTB/WikiText-103
loaders verbatim (same sources, same tokenization with default
`add_special_tokens=True`, same chunking incl. the final partial chunk, first
`N_EVAL_SAMPLES` chunks); teacher-forced step-by-step loop with the same
sync-fenced per-step timing (TTFT = first step, TBT = mean of the rest, total
latency = sum; engine work inside the timed bracket, since eviction is the
method's real per-token cost, just as QuantLinearSim's cost sits inside
KVQuant's timed forwards); GSM8K loaded verbatim (sequential first
`N_EVAL_SAMPLES`, no shuffle), generated with **one batched prefill forward**
(matching `model.generate()`'s prefill in the KVQuant notebooks, and matching
how the official `H2OLlamaAttention` natively handles multi-token prefill:
full attention map in, one prune out), TTFT = prefill until first-token
logits, TBT = (total - TTFT)/(n_generated - 1), grading verbatim (truncate at
`"Question:"`, `#### <num>`-first regex, |pred - gold| < 1e-4); GSM8K
perplexity = the same secondary untimed teacher-forced pass over the gold
final number, aggregated the same way (mean of per-question perplexities);
memory per sample = peak observed cache bytes, dataset peak = max over
samples, average = mean over samples; identical CSV columns saved to the same
Drive folder. On GSM8K the peak includes the transient dense prompt cache
during batched prefill (before the post-prefill prune) -- faithful to
batched-prefill H2O, whose savings are steady-state; the steady-state
post-prune peak is also printed.

Run cells top to bottom. Needs a GPU runtime. Before comparing across
notebooks, confirm the printed GPU name matches the other runs.

In [ ]:
# Block 1 - Environment setup
# Run once per fresh runtime. Package versions are pinned so environment
# differences are never a confound between compression methods -- kept
# byte-for-byte identical to the KVQuant-family notebooks (including
# datasets==2.14.5, which the previous H2O notebook left unpinned).

from google.colab import drive
drive.mount("/content/drive")

!python -m pip install -q --no-deps \
  "transformers==4.43.4" \
  "accelerate==0.33.0" \
  "tokenizers==0.19.1" \
  "huggingface_hub==0.36.2" \
  sentencepiece \
  einops

!python -m pip install -q \
  "datasets==2.14.5" \
  tqdm \
  matplotlib

!python -m pip install -q --no-deps --force-reinstall "huggingface_hub==0.36.2"

import os

try:
    from google.colab import userdata
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    _hf_token = os.environ.get("HF_TOKEN")

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN found -- skipping login (fine for public models like TinyLlama)")

print("Block 1 finished. Now run Block 2.")

In [ ]:
# Block 2 - Imports, GPU check
# Identical to the KVQuant-family notebooks (shared helpers included:
# sync_if_cuda, clear_memory, clear_hf_dataset_cache, robust_call).

import gc
import math
import os
import re
import shutil
import time
import random
import pickle
import sys

import numpy as np
import torch
import torch.nn as nn
import pandas as pd

import datasets
import transformers
import huggingface_hub
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
MODEL_DTYPE = torch.float16 if HAS_CUDA else torch.float32


def sync_if_cuda():
    if HAS_CUDA:
        torch.cuda.synchronize()


def clear_memory():
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()


def clear_hf_dataset_cache(*dataset_names):
    """Removes cached files for the given HF dataset repo name(s) (e.g.
    "wikitext", "gsm8k") from both the datasets cache and the hub cache.
    Used as an on_retry hook: a download that breaks partway through can
    leave a corrupted partial file that every subsequent retry just
    resumes (and re-breaks at the same point) instead of truly restarting
    -- clearing the cache forces a genuinely fresh download."""
    home = os.path.expanduser("~")
    for name in dataset_names:
        for base in [
            os.path.join(home, ".cache", "huggingface", "datasets", name),
            os.path.join(home, ".cache", "huggingface", "hub", f"datasets--{name}"),
        ]:
            shutil.rmtree(base, ignore_errors=True)


def robust_call(fn, *args, max_retries=5, backoff_sec=5, desc="operation", on_retry=None, **kwargs):
    """Retries fn(*args, **kwargs) on any exception, up to max_retries times,
    waiting backoff_sec between attempts -- guards dataset downloads against
    transient network failures (e.g. IncompleteRead/ChunkedEncodingError)
    rather than letting one flaky connection kill the whole notebook run.
    If on_retry is given, it's called (no args) after each failure, before
    the next attempt -- e.g. clear_hf_dataset_cache, so a retry that hit a
    stuck/corrupted partial download actually starts fresh instead of
    resuming (and re-breaking at) the same point every time."""
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            _msg = f"  {desc}: attempt {attempt}/{max_retries} failed ({e!r})"
            if attempt < max_retries:
                _msg += f", retrying in {backoff_sec}s..."
            print(_msg)
            if attempt < max_retries:
                if on_retry is not None:
                    on_retry()
                time.sleep(backoff_sec)
    raise last_err


if not HAS_CUDA:
    print("WARNING: No GPU detected. This will be very slow.")

In [ ]:
# Block 3 - Experiment settings.
# Every shared quantity (MODEL_ID, SHARED_SEED, C, N_EVAL_SAMPLES,
# GSM8K_MAX_NEW_TOKENS, GSM8K few-shot prompt) is byte-for-byte identical to
# the KVQuant-family notebooks, so the methods' evaluations differ only where
# the compression method itself makes them differ. The H2O-specific block at
# the bottom holds the method's own hyperparameters.

LOCAL_MODEL_PATH = "/content/tinyllama-1.1b"
HF_MODEL_ID = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
MODEL_ID = LOCAL_MODEL_PATH if os.path.exists(LOCAL_MODEL_PATH) else HF_MODEL_ID

SHARED_SEED = 0
C = 2048                    # chunk length for PTB/WikiText-103 = model's max context
N_EVAL_SAMPLES = 64          # GSM8K question count; also caps PTB/WikiText-103 source prompts
GSM8K_MAX_NEW_TOKENS = 256
METHOD_NAME = "h2o_budget_60pct"

# ---- H2O method hyperparameters (the only method-specific settings) ----
# The official engine (H2OKVCache_LayerWise, imported from the submodule
# below) takes ABSOLUTE hh_size / recent_size token counts, exactly like the
# repo's run_summarization.py. We derive them per sample from these ratios:
#   hh_size     = int(HEAVY_RATIO  * total_len)
#   recent_size = int(RECENT_RATIO * total_len)
# 60% total budget (1.5x the 40% sibling, 3x the 20% sibling), evenly split
# between heavy-hitter and recency components, same convention as the
# other two.
HEAVY_RATIO = 0.30
RECENT_RATIO = 0.30

random.seed(SHARED_SEED)
torch.manual_seed(SHARED_SEED)

GSM8K_FEWSHOT_PREFIX = (
    "You are solving grade-school math word problems.\n"
    "Show the calculation briefly, then end with exactly this format:\n"
    "#### <final number>\n\n"
    "Question: Mia has 3 marbles and buys 4 more. How many marbles does she have?\n"
    "Answer: Mia has 3 + 4 = 7 marbles.\n"
    "#### 7\n\n"
    "Question: A box has 6 pencils. Sam buys 5 boxes. How many pencils does Sam buy?\n"
    "Answer: Sam buys 6 * 5 = 30 pencils.\n"
    "#### 30\n"
)

print("Model:", MODEL_ID)
print("Method:", METHOD_NAME)
print("Chunk length C:", C)
print("N_EVAL_SAMPLES (GSM8K):", N_EVAL_SAMPLES)
print("GSM8K max new tokens:", GSM8K_MAX_NEW_TOKENS)
print("H2O heavy ratio:", HEAVY_RATIO, "| recent ratio:", RECENT_RATIO,
      "| total budget ratio:", HEAVY_RATIO + RECENT_RATIO)

In [ ]:
# Block - Load tokenizer + the stock (unmodified) model.
# Identical loading code and kwargs to the KVQuant-family notebooks, with ONE
# method-required difference: attn_implementation="eager". H2O needs the real
# per-step attention weights (output_attentions=True) to score heavy hitters,
# and neither FlashAttention nor SDPA can return them -- eager is a hard
# requirement of the method, not a harness choice. Everything else
# (torch_dtype, low_cpu_mem_usage, device_map, use_fast=False tokenizer,
# pad=eos) matches the other notebooks exactly.
#
# NOTE for cross-method latency analysis: because the KVQuant notebooks run
# sdpa, their timing gap vs. this notebook mixes "eager tax" with "H2O tax".
# Run the baseline notebook once with attn_implementation="eager" to separate
# the two.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_model_kwargs = {
    "torch_dtype": MODEL_DTYPE,
    "low_cpu_mem_usage": True,
    "attn_implementation": "eager",   # H2O method requirement (see note above)
    "trust_remote_code": True,
}
if HAS_CUDA:
    _model_kwargs["device_map"] = {"": 0}

print("Loading stock model (H2O modifies the cache at runtime, not the model)...")
model_h2o = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_model_kwargs)
if not HAS_CUDA:
    model_h2o = model_h2o.to(DEVICE)
model_h2o.eval()
model_h2o.config.use_cache = True

device = next(model_h2o.parameters()).device
print("model_h2o loaded on:", device,
      "| dtype:", next(model_h2o.parameters()).dtype,
      "| attn: eager (required by H2O)")

In [ ]:
# Repo setup - Clone the KVCacheCompression repo (always fresh, matching the
# KVQuant notebooks' clean-clone convention) and initialize ONLY the H2O
# submodule -- the official FMInference/H2O repository. The other submodules
# (KVQuant, KIVI, SnapKV, ...) are not needed here, so a targeted init keeps
# this much faster than --recurse-submodules.
#
# From the submodule we import H2OKVCache_LayerWise from
# h2o_hf/utils_real_drop/modify_llama.py: the authors' real-KV-dropping
# eviction engine (per-layer, per-head heavy-hitter scores; physical pruning
# of the cache tensors). The module's other contents (H2OLlamaAttention /
# H2OLlamaForCausalLM) target an older transformers API and MHA-only models,
# so they are not used -- see the notebook intro for details.

if os.path.exists("/content/KVCacheCompression"):
    shutil.rmtree("/content/KVCacheCompression")
    print("Removed existing repo copy for a clean re-clone")

!git clone https://github.com/yoshikodes/KVCacheCompression.git /content/KVCacheCompression
%cd /content/KVCacheCompression
!git submodule update --init --depth 1 H2O

_h2o_module_path = "/content/KVCacheCompression/H2O/h2o_hf/utils_real_drop/modify_llama.py"
assert os.path.exists(_h2o_module_path), \
    "ERROR: official H2O module not found -- clone or submodule init may have failed."

sys.path.insert(0, "/content/KVCacheCompression/H2O/h2o_hf")
from utils_real_drop.modify_llama import H2OKVCache_LayerWise

print("Imported official H2O engine:", H2OKVCache_LayerWise,
      "\nfrom", _h2o_module_path)

In [ ]:
# Block - H2O machinery around the OFFICIAL engine.
# The eviction algorithm itself lives entirely in the submodule's
# H2OKVCache_LayerWise (imported above, unmodified): per-head heavy-hitter
# score accumulation, per-head top-k + recent keep-set selection, physical
# KV dropping. This cell only provides the plumbing that feeds it:
#
#   make_h2o_engines(total_len)
#       One fresh engine per layer (fresh hh_score), exactly as the official
#       H2OLlamaAttention holds one engine per layer. hh_size/recent_size are
#       derived from HEAVY_RATIO/RECENT_RATIO of this sample's total length.
#
#   reduce_attn_to_kv_heads(att)
#       The one GQA adaptation (submodule code untouched): the engine expects
#       one score row per cached KV head, so the 32 query heads' attention
#       weights are summed within each group of 8 sharing a KV head.
#
#   apply_engines(pkv, attentions, engines)
#       Feeds each layer's (legacy KV tuple, reduced attention) through that
#       layer's engine -- the same call H2OLlamaAttention.forward makes
#       (self.kv_cache(past_key_value, attn_weights)) -- and rebuilds the
#       legacy cache.
#
#   h2o_prefill(input_ids, engines)
#       ONE batched forward over the whole prompt (matching the batched
#       prefill inside model.generate() that the KVQuant notebooks use, and
#       matching how the official attention class natively handles
#       multi-token prefill: the full [.., q_len, kv_len] attention map goes
#       into the engine, whose _update_hh_score sums over the query rows),
#       then one engine pass (prune to budget).
#
#   h2o_step(token_tensor, abs_pos, pkv, engines)
#       One decode step: forward one token with explicit position_ids /
#       attention_mask (so RoPE positions stay absolute despite eviction --
#       cached keys were rotated at their original positions before caching,
#       so pruning preserves their correctness), then one engine pass.
#       Handles pkv=None (first step of the teacher-forced loop).

import contextlib
import io

N_LAYERS = model_h2o.config.num_hidden_layers
N_KV_HEADS = int(getattr(model_h2o.config, "num_key_value_heads",
                         model_h2o.config.num_attention_heads))
N_Q_HEADS = model_h2o.config.num_attention_heads
KV_GROUP = N_Q_HEADS // N_KV_HEADS
print(f"Layers: {N_LAYERS} | query heads: {N_Q_HEADS} | KV heads: {N_KV_HEADS} "
      f"| GQA group size: {KV_GROUP}")


def cache_to_legacy(past_key_values):
    if past_key_values is None:
        return None
    if isinstance(past_key_values, tuple):
        return past_key_values
    if hasattr(past_key_values, "to_legacy_cache"):
        return past_key_values.to_legacy_cache()
    raise TypeError(f"Unsupported cache type: {type(past_key_values)}")


def get_cache_tokens(past_key_values):
    past_key_values = cache_to_legacy(past_key_values)
    if past_key_values is None:
        return 0
    return int(past_key_values[0][0].shape[2])


def dense_kv_cache_bytes(past_key_values):
    """Real bytes of the tensors actually resident in past_key_values right
    now, at their current dtype. The official engine physically drops
    positions, so this is an honest MEASURED number (the quantized KVQuant
    notebooks, by contrast, report CALCULATED bytes because their
    compression is simulated)."""
    past_key_values = cache_to_legacy(past_key_values)
    if past_key_values is None:
        return 0
    total = 0
    for key, value in past_key_values:
        total += key.numel() * key.element_size()
        total += value.numel() * value.element_size()
    return int(total)


def kv_bytes_per_token(past_key_values):
    """Exact real bytes one cached token occupies (uniform across positions:
    fp16 K + fp16 V for every layer). Used to convert a tracked max cache
    length into peak bytes without touching the GPU inside timed regions."""
    n_tokens = get_cache_tokens(past_key_values)
    if n_tokens == 0:
        return 0.0
    return dense_kv_cache_bytes(past_key_values) / n_tokens


def make_h2o_engines(total_len):
    """One fresh official engine per layer, budgets from this sample's total
    length. The engine's constructor print ("H2OKVCache-LayerWise: hh, rec")
    is suppressed -- it would fire n_layers times per sample; the budgets are
    reported once by the callers instead."""
    hh_size = max(1, int(total_len * HEAVY_RATIO))
    recent_size = max(1, int(total_len * RECENT_RATIO))
    with contextlib.redirect_stdout(io.StringIO()):
        engines = [
            H2OKVCache_LayerWise(hh_size=hh_size, recent_size=recent_size,
                                 k_seq_dim=2, v_seq_dim=2)
            for _ in range(N_LAYERS)
        ]
    return engines


def reduce_attn_to_kv_heads(att):
    """[batch, n_q_heads, q_len, kv_len] -> [batch, n_kv_heads, q_len, kv_len]
    by summing the query heads within each GQA group: the total attention
    mass received by that KV head's cache entries. (In HF Llama, query head h
    reads KV head h // KV_GROUP, i.e. groups are contiguous blocks.) On MHA
    models KV_GROUP == 1 and this is the identity."""
    if KV_GROUP == 1:
        return att
    b, h, q, kv = att.shape
    return att.view(b, N_KV_HEADS, KV_GROUP, q, kv).sum(dim=2)


def apply_engines(past_key_values, attentions, engines):
    """One official-engine call per layer -- the same call the official
    H2OLlamaAttention.forward makes: self.kv_cache(past_key_value,
    attn_weights). Updates that layer's per-head hh_score and physically
    prunes that layer's KV if over budget."""
    past_key_values = cache_to_legacy(past_key_values)
    new_layers = []
    for layer_kv, layer_att, engine in zip(past_key_values, attentions, engines):
        att_kv = reduce_attn_to_kv_heads(layer_att.detach())
        pruned = engine(tuple(layer_kv), att_kv)
        new_layers.append(tuple(pruned))
    return tuple(new_layers)


@torch.no_grad()
def h2o_prefill(input_ids, engines):
    """Returns (last_logits, pruned_pkv, prefill_cache_tokens).
    prefill_cache_tokens is the cache length BEFORE the post-prefill prune --
    the transient dense peak that batched-prefill H2O genuinely incurs."""
    outputs = model_h2o(
        input_ids=input_ids,
        use_cache=True,
        output_attentions=True,
        return_dict=True,
    )
    last_logits = outputs.logits[:, -1, :]
    pkv = cache_to_legacy(outputs.past_key_values)
    prefill_cache_tokens = get_cache_tokens(pkv)
    pkv = apply_engines(pkv, outputs.attentions, engines)
    return last_logits, pkv, prefill_cache_tokens


@torch.no_grad()
def h2o_step(token_tensor, abs_pos, past_key_values, engines):
    """One decode step through the H2O-compressed cache."""
    cache_len_before = get_cache_tokens(past_key_values)

    attention_mask = torch.ones(
        (1, cache_len_before + 1),
        dtype=torch.long,
        device=device,
    )
    position_ids = torch.tensor([[abs_pos]], dtype=torch.long, device=device)

    outputs = model_h2o(
        input_ids=token_tensor,
        past_key_values=past_key_values,
        attention_mask=attention_mask,
        position_ids=position_ids,
        use_cache=True,
        output_attentions=True,
        return_dict=True,
    )

    past_key_values = apply_engines(
        outputs.past_key_values, outputs.attentions, engines,
    )
    return outputs.logits[:, -1, :], past_key_values


_demo = make_h2o_engines(C)
print("Official H2O engines ready. Example budgets at total_len =", C, ":",
      f"hh_size {_demo[0].hh_size}, recent_size {_demo[0].recent_size},",
      f"cache_size {_demo[0].cache_size} tokens",
      f"({(_demo[0].cache_size / C):.1%} of dense)")
del _demo

In [ ]:
# Block - PTB / WikiText-103: load the FULL test text, concatenate ALL of
# it into one long sequence, tokenize, split into consecutive chunks of
# length C, then cap to the FIRST N_EVAL_SAMPLES chunks -- a deterministic
# prefix of the real test set, not a random sample.
#
# This cell is VERBATIM from the KVQuant-family notebooks (same PTB raw-file
# source, same unstripped WikiText-103 lines, same "\n\n" join, same default
# add_special_tokens=True tokenization, same chunk_sequence that keeps the
# final partial chunk) so all five methods are evaluated on token-for-token
# identical data. Both downloads go through robust_call so a transient
# network blip gets retried instead of killing the whole run; WikiText-103's
# load also clears its HF cache between retries (on_retry), since a broken
# 157MB download can leave a corrupted partial file that plain retries
# just resume (and re-break at the same point) forever.


def chunk_sequence(token_ids_1d, chunk_len):
    chunks = []
    n = token_ids_1d.shape[0]
    for start in range(0, n, chunk_len):
        chunks.append(token_ids_1d[start:start + chunk_len])
    return chunks


def load_ptb_chunks():
    import urllib.request
    test_url = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.test.txt"

    def _fetch():
        return urllib.request.urlopen(test_url).read().decode("utf-8")

    test_text_raw = robust_call(_fetch, desc="PTB test.txt download")
    sentences = [s.strip() for s in test_text_raw.splitlines() if s.strip()]
    full_text = "\n\n".join(sentences)
    ids = tokenizer(full_text, return_tensors="pt")["input_ids"][0]
    print(f"PTB test set: {len(sentences)} lines, {ids.shape[0]} tokens total")
    return chunk_sequence(ids, C)[:N_EVAL_SAMPLES]


def load_wikitext103_chunks():
    testdata = robust_call(
        load_dataset, "wikitext", "wikitext-103-raw-v1", split="test",
        desc="WikiText-103 test load", on_retry=lambda: clear_hf_dataset_cache("wikitext"),
    )
    texts = [t for t in testdata["text"] if len(t.strip()) > 0]
    full_text = "\n\n".join(texts)
    ids = tokenizer(full_text, return_tensors="pt")["input_ids"][0]
    print(f"WikiText-103 test set: {len(texts)} non-empty lines, {ids.shape[0]} tokens total")
    return chunk_sequence(ids, C)[:N_EVAL_SAMPLES]


ptb_chunks = load_ptb_chunks()
wikitext103_chunks = load_wikitext103_chunks()
print(f"PTB: {len(ptb_chunks)} chunks of up to {C} tokens (first N_EVAL_SAMPLES={N_EVAL_SAMPLES}, not randomly selected)")
print(f"WikiText-103: {len(wikitext103_chunks)} chunks of up to {C} tokens (first N_EVAL_SAMPLES={N_EVAL_SAMPLES}, not randomly selected)")

In [ ]:
# Block - Step-by-step-per-chunk eval function.
#
# Mirrors the KVQuant notebooks' score_chunk_kvquant exactly: the model is
# stepped one token at a time using its own KV cache, teacher-forced (real
# corpus tokens fed in, never the model's own prediction), each step timed
# between the same cuda.synchronize() fences, NLL via the same
# CrossEntropyLoss on the step logits, next-token accuracy via argmax match.
# TTFT = the first step, TBT = mean of the remaining steps, total latency =
# sum of steps -- identical definitions and identical loop structure.
#
# The one method difference: each step goes through h2o_step, so the official
# engines' scoring + eviction happen INSIDE the timed bracket. That's
# deliberate: eviction is H2O's real per-token cost, exactly as
# QuantLinearSim's quantize/dequantize cost sits inside the KVQuant
# notebooks' timed forward.
#
# Fresh official engines (fresh per-head hh_scores) are created per chunk,
# mirroring the official per-request _clean_scores usage; budgets come from
# this chunk's length via the shared ratios.
#
# Memory: the engines physically prune the cache, so the per-chunk value is
# the MEASURED peak -- max cache length seen across steps (a cheap shape read
# done OUTSIDE the timing fences, so it cannot affect any timing number)
# converted to real fp16 tensor bytes.


@torch.no_grad()
def score_chunk_h2o(chunk_ids_1d):
    L = chunk_ids_1d.shape[0]
    if L < 2:
        return None
    chunk_ids = chunk_ids_1d.unsqueeze(0).to(device)
    input_ids = chunk_ids[:, :-1]
    target_ids = chunk_ids[:, 1:]

    engines = make_h2o_engines(L)

    loss_fct = nn.CrossEntropyLoss()
    nll_sum = 0.0
    correct = 0
    scored = 0
    step_times = []
    past_key_values = None
    max_cache_tokens = 0

    for pos in range(input_ids.shape[1]):
        step_input = input_ids[:, pos:pos + 1]

        sync_if_cuda()
        t0 = time.perf_counter()
        step_logits, past_key_values = h2o_step(
            step_input, pos, past_key_values, engines,
        )
        sync_if_cuda()
        step_times.append(time.perf_counter() - t0)

        step_target = target_ids[:, pos]

        loss = loss_fct(step_logits, step_target)
        nll_sum += loss.float().item()

        pred = step_logits.argmax(dim=-1)
        correct += int((pred == step_target).sum().item())
        scored += 1

        # Shape read only -- outside the timing fences, no GPU work.
        max_cache_tokens = max(max_cache_tokens, get_cache_tokens(past_key_values))

    ttft_sec = step_times[0]
    tbt_sec = sum(step_times[1:]) / len(step_times[1:]) if len(step_times) > 1 else 0.0
    total_latency_sec = sum(step_times)

    peak_bytes = int(round(max_cache_tokens * kv_bytes_per_token(past_key_values)))

    return {
        "nll_sum": nll_sum, "scored": scored, "correct": correct,
        "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "peak_memory_bytes": peak_bytes,
        "max_cache_tokens": max_cache_tokens,
    }


@torch.no_grad()
def preview_chunk_prediction(chunk_ids_1d, n_preview_tokens=30):
    """Untimed, separate pass over just the first n_preview_tokens of a chunk
    -- for printing what the model actually predicted vs. the real next
    tokens. Runs through the same h2o_step machinery (with fresh engines
    budgeted from the full chunk length, same as the scored run) so the
    preview reflects the method's behavior. Does not affect any reported
    metric."""
    n_preview_tokens = min(n_preview_tokens, chunk_ids_1d.shape[0] - 1)
    if n_preview_tokens < 1:
        return None
    L = chunk_ids_1d.shape[0]
    engines = make_h2o_engines(L)
    preview_ids = chunk_ids_1d[:n_preview_tokens + 1].unsqueeze(0).to(device)
    input_ids = preview_ids[:, :-1]
    target_ids = preview_ids[:, 1:]

    past_key_values = None
    preds = []
    for pos in range(input_ids.shape[1]):
        step_logits, past_key_values = h2o_step(
            input_ids[:, pos:pos + 1], pos, past_key_values, engines,
        )
        preds.append(int(step_logits.argmax(dim=-1)[0].item()))

    return {
        "prompt_text": tokenizer.decode(input_ids[0], skip_special_tokens=True),
        "actual_next_text": tokenizer.decode(target_ids[0], skip_special_tokens=True),
        "predicted_next_text": tokenizer.decode(preds, skip_special_tokens=True),
    }

In [ ]:
# Block - PTB / WikiText-103 driver: run every chunk through score_chunk_h2o.
# Aggregation is identical to the KVQuant notebooks: perplexity =
# exp(pooled mean NLL over all scored tokens), accuracy = pooled next-token
# accuracy, TTFT/TBT/latency = means over chunks, peak_memory_gb = max over
# chunks, average_memory_gb = mean over chunks. Same output columns.


def evaluate_lm_dataset_h2o(dataset_name, chunks, method_label):
    clear_memory()
    total_nll = 0.0
    total_scored = 0
    total_correct = 0
    ttft_values, tbt_values, latency_values, peak_mem_values = [], [], [], []
    max_cache_tokens_seen = 0

    N_PREVIEW_CHUNKS = 5
    for chunk_idx, chunk_ids in enumerate(tqdm(chunks, desc=f"{dataset_name} | {method_label}")):
        if chunk_idx < N_PREVIEW_CHUNKS:
            preview = preview_chunk_prediction(chunk_ids)
            if preview is not None:
                print(f"\n--- {dataset_name} | {method_label} | chunk {chunk_idx} preview ---")
                print(f"Prompt:              {preview['prompt_text']!r}")
                print(f"Actual next text:    {preview['actual_next_text']!r}")
                print(f"Predicted next text: {preview['predicted_next_text']!r}")

        result = score_chunk_h2o(chunk_ids)
        if result is None:
            continue
        total_nll += result["nll_sum"]
        total_scored += result["scored"]
        total_correct += result["correct"]
        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        max_cache_tokens_seen = max(max_cache_tokens_seen, result["max_cache_tokens"])

    avg_nll = total_nll / max(total_scored, 1)
    perplexity = math.exp(min(avg_nll, 50.0))
    accuracy = total_correct / max(total_scored, 1)

    print(f"\n{dataset_name}: max retained cache across chunks = {max_cache_tokens_seen} tokens "
          f"(budget ratio {HEAVY_RATIO + RECENT_RATIO}; memory is MEASURED from the physically pruned cache)")

    return {
        "dataset": dataset_name,
        "method": method_label,
        "perplexity": perplexity,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


lm_results = []
for _name, _chunks in [("PTB", ptb_chunks), ("WikiText-103", wikitext103_chunks)]:
    lm_results.append(evaluate_lm_dataset_h2o(_name, _chunks, METHOD_NAME))

lm_results_df = pd.DataFrame(lm_results)
display(lm_results_df)

In [ ]:
# Block - GSM8K loading: canonical source, sequential first N_EVAL_SAMPLES
# questions (no shuffling), few-shot prompt -- VERBATIM from the
# KVQuant-family notebooks so all five methods see the exact same question
# set and prompts. Download goes through robust_call, clearing its HF cache
# between retries, so a transient network blip or stuck partial download
# gets retried cleanly instead of killing the whole run.


def extract_gsm8k_gold_answer(answer_text):
    m = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", answer_text)
    if not m:
        return None
    try:
        return float(m.group(1).replace(",", ""))
    except ValueError:
        return None


gsm8k_test = robust_call(
    load_dataset, "gsm8k", "main", split="test",
    desc="GSM8K test load", on_retry=lambda: clear_hf_dataset_cache("gsm8k"),
)

gsm8k_qa_pairs = []
for i in range(min(N_EVAL_SAMPLES, len(gsm8k_test))):
    item = gsm8k_test[i]
    gold = extract_gsm8k_gold_answer(item["answer"])
    if gold is not None:
        gsm8k_qa_pairs.append({"question": item["question"], "gold": gold, "gold_text": item["answer"]})

print(f"GSM8K: {len(gsm8k_qa_pairs)} questions loaded (of {len(gsm8k_test)} total in test set)")

In [ ]:
# Block - GSM8K generation with H2O (official engines). Prefill processes the
# whole prompt in ONE batched forward (exactly like the prefill inside
# model.generate() that the KVQuant notebooks use, and exactly how the
# official H2OLlamaAttention handles multi-token inputs: the full
# [.., q_len, kv_len] attention map goes into the engine, whose
# _update_hh_score sums over the query rows); decode then continues greedily
# one token at a time with per-step eviction.
#
# Timing matches the KVQuant notebooks' definitions exactly:
#   TTFT  = time from generation start until the first generated token's
#           logits are ready (i.e., the batched prefill forward) -- the same
#           quantity their StoppingCriteria timestamp captures.
#   total = wall time of the whole generation.
#   TBT   = (total - TTFT) / max(n_generated - 1, 1) -- the same formula.
# The post-prefill engine pass lands AFTER the TTFT timestamp (the first
# token comes straight from the prefill logits and does not need the prune),
# so it is counted in total latency / TBT as decode-phase method overhead.
#
# n_generated counts every token an actual timed forward call produced
# (prefill's token + one per h2o_step call), INCLUDING a final EOS token
# that ends the loop -- matching the KVQuant notebooks' StoppingCriteria,
# which fires after the EOS-producing step completes too, before
# generate() checks for termination. generated_ids/gen_text deliberately
# excludes EOS itself (matching skip_special_tokens=True decoding
# elsewhere), but n_generated must still count that step's forward call,
# or tbt_sec's denominator undercounts by one relative to KVQuant every
# time a question ends via EOS rather than hitting GSM8K_MAX_NEW_TOKENS --
# tracked separately below as n_forward_tokens.
#
# Answer grading is VERBATIM from the KVQuant notebooks: truncate the
# generation at the first "Question:", then extract "#### <num>" (falling
# back to the last number), compare |pred - gold| < 1e-4.
#
# Memory: MEASURED peak = max cache length observed (tracked via cheap shape
# reads), converted to real fp16 tensor bytes. NOTE: with batched prefill the
# peak includes the transient moment when the full dense prompt cache exists
# before the post-prefill prune -- that is genuinely how batched-prefill H2O
# behaves (its savings are steady-state / decode-phase). The steady-state
# post-prune size is also returned for reference and printed by the driver.


def _extract_final_number(text):
    m = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", text)
    if m:
        num_str = m.group(1)
    else:
        nums = re.findall(r"-?[0-9][0-9,]*\.?[0-9]*", text)
        if not nums:
            return None
        num_str = nums[-1]
    num_str = num_str.replace(",", "").rstrip(".")
    try:
        return float(num_str)
    except ValueError:
        return None


@torch.no_grad()
def generate_gsm8k_h2o(question):
    prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {question.strip()}\nAnswer:"
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]

    engines = make_h2o_engines(prompt_len + GSM8K_MAX_NEW_TOKENS)

    sync_if_cuda()
    gen_start = time.perf_counter()

    # ---- Prefill (one batched forward; first token's logits ready here).
    # Done inline rather than via h2o_prefill so the TTFT timestamp can land
    # between the forward and the engine pass -- see header comment. ----
    outputs = model_h2o(input_ids=enc["input_ids"], use_cache=True,
                        output_attentions=True, return_dict=True)
    last_logits = outputs.logits[:, -1, :]
    next_id = int(last_logits.argmax(dim=-1)[0].item())
    sync_if_cuda()
    ttft_sec = time.perf_counter() - gen_start

    # ---- Official-engine pass on the prefill attention map, then prune
    # (counted in total latency, not TTFT -- see header comment) ----
    pkv = cache_to_legacy(outputs.past_key_values)
    prefill_cache_tokens = get_cache_tokens(pkv)
    pkv = apply_engines(pkv, outputs.attentions, engines)

    max_cache_tokens = prefill_cache_tokens
    steady_state_max_cache_tokens = get_cache_tokens(pkv)
    generated_ids = []
    n_forward_tokens = 1  # the prefill forward already produced next_id -- see header comment

    # ---- Greedy decode with per-step eviction ----
    for step in range(GSM8K_MAX_NEW_TOKENS):
        if next_id == tokenizer.eos_token_id:
            break
        generated_ids.append(next_id)
        if step == GSM8K_MAX_NEW_TOKENS - 1:
            break  # no wasted forward for a token we would never use

        token_tensor = torch.tensor([[next_id]], dtype=torch.long, device=device)
        abs_pos = prompt_len + step
        last_logits, pkv = h2o_step(token_tensor, abs_pos, pkv, engines)
        next_id = int(last_logits.argmax(dim=-1)[0].item())
        n_forward_tokens += 1

        cache_tokens = get_cache_tokens(pkv)  # shape read only
        max_cache_tokens = max(max_cache_tokens, cache_tokens)
        steady_state_max_cache_tokens = max(steady_state_max_cache_tokens, cache_tokens)

    sync_if_cuda()
    gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    gen_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    gen_text = gen_text.split("Question:")[0]

    n_generated = n_forward_tokens
    if len(generated_ids) == 0:
        ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)

    total_tokens = prompt_len + n_generated

    bpt = kv_bytes_per_token(pkv)
    peak_bytes = int(round(max_cache_tokens * bpt))
    steady_state_bytes = int(round(steady_state_max_cache_tokens * bpt))

    return {
        "gen_text": gen_text, "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "total_tokens": total_tokens,
        "peak_memory_bytes": peak_bytes,
        "steady_state_memory_bytes": steady_state_bytes,
        "prefill_cache_tokens": prefill_cache_tokens,
        "steady_state_max_cache_tokens": steady_state_max_cache_tokens,
    }


@torch.no_grad()
def score_gsm8k_perplexity_h2o(question, gold_text):
    """Secondary, untimed teacher-forced pass on the gold answer -- same
    definition as the KVQuant notebooks (perplexity of the gold final number
    given the prompt), with the method applied: the prompt goes through the
    batched prefill + official-engine prune, and the gold-answer tokens are
    then fed step-by-step through the H2O-compressed cache."""
    gold_number = extract_gsm8k_gold_answer(gold_text)
    if gold_number is None:
        return None
    gold_str = str(int(gold_number)) if gold_number == int(gold_number) else str(gold_number)
    prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {question.strip()}\nAnswer:"
    prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
    target_ids = tokenizer(" " + gold_str, add_special_tokens=False)["input_ids"]

    engines = make_h2o_engines(len(prompt_ids) + len(target_ids))
    input_ids = torch.tensor([prompt_ids], device=device)

    last_logits, pkv, _ = h2o_prefill(input_ids, engines)

    nll_sum = 0.0
    scored = 0
    for ti, target_id in enumerate(target_ids):
        log_probs = torch.log_softmax(last_logits[0].float(), dim=-1)
        nll_sum += -log_probs[target_id].item()
        scored += 1
        if ti < len(target_ids) - 1:
            token_tensor = torch.tensor([[target_id]], dtype=torch.long, device=device)
            abs_pos = len(prompt_ids) + ti
            last_logits, pkv = h2o_step(token_tensor, abs_pos, pkv, engines)
    if scored == 0:
        return None
    return math.exp(min(nll_sum / scored, 50.0))

In [ ]:
# Block - GSM8K driver: run every question through generate_gsm8k_h2o
# (accuracy + TTFT/TBT/latency + memory, all from the SAME real generation
# call), plus the secondary teacher-forced perplexity pass. Aggregation is
# identical to the KVQuant notebooks: accuracy = fraction correct,
# perplexity = MEAN of per-question perplexities, TTFT/TBT/latency = means,
# peak_memory_gb = max over questions, average_memory_gb = mean over
# questions. Same output columns.


def evaluate_gsm8k_h2o(qa_pairs, method_label):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    steady_state_mem_values = []

    N_PREVIEW_QUESTIONS = 5
    for q_idx, qa in enumerate(tqdm(qa_pairs, desc=f"GSM8K | {method_label}")):
        result = generate_gsm8k_h2o(qa["question"])
        pred = _extract_final_number(result["gen_text"])
        is_correct = pred is not None and abs(pred - qa["gold"]) < 1e-4
        correct += int(is_correct)
        total += 1

        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- GSM8K | {method_label} | question {q_idx} preview ---")
            print(f"Question:    {qa['question']}")
            print(f"Generated:   {result['gen_text'].strip()}")
            print(f"Gold answer: {qa['gold']} | Predicted: {pred} | Correct: {is_correct}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        steady_state_mem_values.append(result["steady_state_memory_bytes"])

        ppl = score_gsm8k_perplexity_h2o(qa["question"], qa["gold_text"])
        if ppl is not None:
            ppl_values.append(ppl)

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    if steady_state_mem_values:
        print(f"\nGSM8K memory note: peak includes the transient dense prompt cache during "
              f"batched prefill (before the post-prefill prune). Steady-state post-prune "
              f"peak = {max(steady_state_mem_values) / 1024**3:.6f} GB, "
              f"avg = {sum(steady_state_mem_values) / len(steady_state_mem_values) / 1024**3:.6f} GB.")

    return {
        "dataset": "GSM8K",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


gsm8k_results = [
    evaluate_gsm8k_h2o(gsm8k_qa_pairs, METHOD_NAME),
]
gsm8k_results_df = pd.DataFrame(gsm8k_results)
display(gsm8k_results_df)

In [ ]:
# Block - Combine PTB / WikiText-103 / GSM8K results into one table with
# ONLY the specified metrics, then save to CSV -- identical column schema to
# the KVQuant-family notebooks, saved into the same Drive folder, so all
# five methods' CSVs concatenate directly into one comparison table.
#
# Cross-method comparison reminders:
#   - This notebook's memory numbers are MEASURED (physically pruned cache);
#     the quantized KVQuant notebooks' are CALCULATED (simulated
#     compression). Say so when presenting them side by side.
#   - This notebook runs eager attention (H2O method requirement); the
#     KVQuant notebooks run sdpa. To decompose "eager tax" vs "H2O tax" in
#     the latency columns, also run the baseline notebook once with
#     attn_implementation="eager".
#   - Confirm the GPU name printed in Block 2 matches every other notebook's
#     run before trusting any timing/memory comparison.

results_df = pd.concat([lm_results_df, gsm8k_results_df], ignore_index=True)
results_df = results_df[[
    "dataset", "method", "perplexity", "accuracy",
    "ttft_sec", "tbt_sec", "avg_total_latency_sec",
    "peak_memory_gb", "average_memory_gb",
]]
display(results_df)

os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)

_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{METHOD_NAME}_results.csv"
results_df.to_csv(_path, index=False)
print(f"Saved to {_path}")